In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\keebs\OneDrive\Documentos\INTELIGENCIA ARTIFICIAL\MATERIAS 5S\PRACTICA PROFESIONALIZANTE 2\pf\bekantor-demand-intelligence


In [2]:
from src.data.load_data import load_raw_data
from src.data.clean_data import minimal_cleaning
from src.data.panel_diagnostics import (
    add_store_operation_flags,
    add_zero_taxonomy,
)

datasets = load_raw_data()
datasets = minimal_cleaning(datasets)

sales = datasets["sales"]

sales, apertura = add_store_operation_flags(sales)
sales = add_zero_taxonomy(sales)

sales_modelable = sales[
    sales["tienda_operativa"].eq(1)
    & sales["familia_surtida"].eq(1)
].copy()

print(f"Panel modelable: {len(sales_modelable):,}")
print(
    sales_modelable["date"].min().date(),
    "→",
    sales_modelable["date"].max().date(),
)

Panel modelable: 2,697,207
2013-01-01 → 2017-08-15


In [3]:
from src.models.validation import temporal_train_valid_split

train, validation, validation_start, validation_end = (
    temporal_train_valid_split(
        sales_modelable,
        validation_days=15,
    )
)

print("TRAIN")
print(train["date"].min().date(), "→", train["date"].max().date())
print(f"Registros: {len(train):,}")

print("\nVALIDACIÓN")
print(validation_start.date(), "→", validation_end.date())
print(f"Registros: {len(validation):,}")
print(f"Días: {validation['date'].nunique()}")

TRAIN
2013-01-01 → 2017-07-31
Registros: 2,671,272

VALIDACIÓN
2017-08-01 → 2017-08-15
Registros: 25,935
Días: 15


In [4]:
assert train["date"].max() < validation["date"].min()
assert validation["date"].nunique() == 15
assert len(train) + len(validation) == len(sales_modelable)

print("✓ Split temporal correcto")
print("✓ Sin solapamiento")
print("✓ 15 días de validación")

✓ Split temporal correcto
✓ Sin solapamiento
✓ 15 días de validación


In [5]:
from src.features.model_features import add_known_features

features_base = add_known_features(
    sales_modelable=sales_modelable,
    sales=sales,
    stores=datasets["stores"],
    holidays=datasets["holidays"],
)

print(f"Filas originales: {len(sales_modelable):,}")
print(f"Filas con features: {len(features_base):,}")

features_base.head()

Filas originales: 2,697,207
Filas con features: 2,697,207


,id,date,store_nbr,family,sales,onpromotion,tienda_operativa,familia_surtida,tipo_cero,year,...,type,cluster,is_holiday,is_event,is_transfer,is_additional,is_bridge,is_work_day,is_new_year_closure,event_count
0,581,2013-01-01,25,LAWN AND GARDEN,2.0,0,True,True,Venta positiva,2013,...,D,1,1,0,0,0,0,0,1,1
1,579,2013-01-01,25,HOME CARE,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
2,578,2013-01-01,25,HOME APPLIANCES,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
3,577,2013-01-01,25,HOME AND KITCHEN II,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
4,576,2013-01-01,25,HOME AND KITCHEN I,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1


In [6]:
assert len(features_base) == len(sales_modelable)

assert not features_base[
    ["city", "state", "type", "cluster"]
].isna().any().any()

assert not features_base[
    [
        "is_holiday",
        "is_event",
        "is_transfer",
        "is_additional",
        "is_bridge",
        "is_work_day",
        "is_new_year_closure",
    ]
].isna().any().any()

print("✓ Cantidad de filas preservada")
print("✓ Metadata de tiendas completa")
print("✓ Calendario completo")
print("✓ Sin utilizar ventas futuras")

✓ Cantidad de filas preservada
✓ Metadata de tiendas completa
✓ Calendario completo
✓ Sin utilizar ventas futuras


In [7]:
features_base.columns.tolist()

['id',
 'date',
 'store_nbr',
 'family',
 'sales',
 'onpromotion',
 'tienda_operativa',
 'familia_surtida',
 'tipo_cero',
 'year',
 'month',
 'day',
 'day_of_week',
 'week_of_year',
 'is_weekend',
 'city',
 'state',
 'type',
 'cluster',
 'is_holiday',
 'is_event',
 'is_transfer',
 'is_additional',
 'is_bridge',
 'is_work_day',
 'is_new_year_closure',
 'event_count']

In [8]:
from src.features.model_features import add_lag_features

features_lags = add_lag_features(
    features_base,
    lags=(1, 7, 14, 28),
)

features_lags[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
    ]
].head(40)

,date,store_nbr,family,sales,lag_1,lag_7,lag_14,lag_28
0,2013-01-01,25,LAWN AND GARDEN,2.000000,NaN,NaN,NaN,NaN
1,2013-01-01,25,HOME CARE,0.000000,NaN,NaN,NaN,NaN
2,2013-01-01,25,HOME APPLIANCES,0.000000,NaN,NaN,NaN,NaN
3,2013-01-01,25,HOME AND KITCHEN II,0.000000,NaN,NaN,NaN,NaN
4,2013-01-01,25,HOME AND KITCHEN I,0.000000,NaN,NaN,NaN,NaN
5,2013-01-01,25,HARDWARE,0.000000,NaN,NaN,NaN,NaN
6,2013-01-01,25,GROCERY II,15.000000,NaN,NaN,NaN,NaN
7,2013-01-01,25,GROCERY I,700.000000,NaN,NaN,NaN,NaN
8,2013-01-01,25,FROZEN FOODS,29.654999,NaN,NaN,NaN,NaN
9,2013-01-01,25,EGGS,46.000000,NaN,NaN,NaN,NaN


In [9]:
assert len(features_lags) == len(features_base)

for lag in [1, 7, 14, 28]:
    assert f"lag_{lag}" in features_lags.columns

print("✓ Filas preservadas")
print("✓ lag_1 creado")
print("✓ lag_7 creado")
print("✓ lag_14 creado")
print("✓ lag_28 creado")

features_lags[
    ["lag_1", "lag_7", "lag_14", "lag_28"]
].isna().mean()

✓ Filas preservadas
✓ lag_1 creado
✓ lag_7 creado
✓ lag_14 creado
✓ lag_28 creado


lag_1     0.003028
lag_7     0.006875
lag_14    0.011362
lag_28    0.020336
dtype: float64

In [10]:
from src.features.model_features import add_rolling_features

features_full = add_rolling_features(
    features_lags,
    windows=(7, 14, 28),
)

features_full[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "lag_7",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
    ]
].head(40)

,date,store_nbr,family,sales,lag_7,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,2013-01-01,25,LAWN AND GARDEN,2.000000,NaN,NaN,NaN,NaN
1,2013-01-01,25,HOME CARE,0.000000,NaN,NaN,NaN,NaN
2,2013-01-01,25,HOME APPLIANCES,0.000000,NaN,NaN,NaN,NaN
3,2013-01-01,25,HOME AND KITCHEN II,0.000000,NaN,NaN,NaN,NaN
4,2013-01-01,25,HOME AND KITCHEN I,0.000000,NaN,NaN,NaN,NaN
5,2013-01-01,25,HARDWARE,0.000000,NaN,NaN,NaN,NaN
6,2013-01-01,25,GROCERY II,15.000000,NaN,NaN,NaN,NaN
7,2013-01-01,25,GROCERY I,700.000000,NaN,NaN,NaN,NaN
8,2013-01-01,25,FROZEN FOODS,29.654999,NaN,NaN,NaN,NaN
9,2013-01-01,25,EGGS,46.000000,NaN,NaN,NaN,NaN


In [11]:
assert len(features_full) == len(features_lags)

for window in [7, 14, 28]:
    assert (
        f"rolling_mean_{window}"
        in features_full.columns
    )

print("✓ Filas preservadas")
print("✓ rolling_mean_7 creada")
print("✓ rolling_mean_14 creada")
print("✓ rolling_mean_28 creada")
print("✓ Día actual excluido")

features_full[
    [
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
    ]
].isna().mean()

✓ Filas preservadas
✓ rolling_mean_7 creada
✓ rolling_mean_14 creada
✓ rolling_mean_28 creada
✓ Día actual excluido


rolling_mean_7     0.000641
rolling_mean_14    0.000641
rolling_mean_28    0.000641
dtype: float64

In [12]:
sample = (
    features_full[
        features_full["rolling_mean_7"].notna()
    ]
    .iloc[10000]
)

mask = (
    features_base["store_nbr"].eq(
        sample["store_nbr"]
    )
    & features_base["family"].eq(
        sample["family"]
    )
    & features_base["date"].ge(
        sample["date"]
        - pd.Timedelta(days=7)
    )
    & features_base["date"].lt(
        sample["date"]
    )
)

manual_mean = features_base.loc[
    mask,
    "sales",
].mean()

print("Rolling calculada:", sample["rolling_mean_7"])
print("Rolling manual:", manual_mean)

assert np.isclose(
    sample["rolling_mean_7"],
    manual_mean,
)

print("✓ Rolling verificada sin usar el día actual")

Rolling calculada: 0.0
Rolling manual: 0.0
✓ Rolling verificada sin usar el día actual


In [13]:
series_cols = ["store_nbr", "family"]

train_positive_series = (
    train.loc[train["sales"] > 0, series_cols]
    .drop_duplicates()
)

validation_positive_series = (
    validation.loc[validation["sales"] > 0, series_cols]
    .drop_duplicates()
)

validation_new_series = (
    validation_positive_series
    .merge(
        train_positive_series,
        on=series_cols,
        how="left",
        indicator=True,
    )
)

validation_new_series = validation_new_series[
    validation_new_series["_merge"].eq("left_only")
]

print(
    "Series con primera demanda positiva "
    "recién en validación:",
    len(validation_new_series),
)

validation_new_series.head()

Series con primera demanda positiva recién en validación: 0


,store_nbr,family,_merge


In [14]:
assert len(validation_new_series) == 0

print("✓ Todas las series con demanda en validación ya eran conocidas en train")
print("✓ La definición del panel no introduce nuevas series desde el futuro")

✓ Todas las series con demanda en validación ya eran conocidas en train
✓ La definición del panel no introduce nuevas series desde el futuro


In [15]:
from src.models.baseline import (
    seasonal_weekday_baseline,
    rmsle,
)

baseline_result = seasonal_weekday_baseline(
    train=train,
    validation=validation,
    lookback_days=28,
)

baseline_result[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "pred_baseline",
    ]
].head()

,date,store_nbr,family,sales,pred_baseline
0,2017-08-01,41,PRODUCE,2037.918,1780.99475
1,2017-08-01,41,SCHOOL AND OFFICE SUPPLIES,0.000,1.50000
2,2017-08-01,41,SEAFOOD,2.224,5.00000
3,2017-08-01,42,AUTOMOTIVE,2.000,4.50000
4,2017-08-01,42,BREAD/BAKERY,643.267,427.97750


In [16]:
baseline_rmsle = rmsle(
    baseline_result["sales"],
    baseline_result["pred_baseline"],
)

baseline_mae = np.mean(
    np.abs(
        baseline_result["sales"]
        - baseline_result["pred_baseline"]
    )
)

print(f"RMSLE baseline: {baseline_rmsle:.4f}")
print(f"MAE baseline:   {baseline_mae:.4f}")

RMSLE baseline: 0.5366
MAE baseline:   85.4780


In [17]:
assert len(baseline_result) == len(validation)

assert baseline_result["pred_baseline"].notna().all()

assert (baseline_result["pred_baseline"] >= 0).all()

print("✓ Todas las filas de validación tienen predicción")
print("✓ Sin predicciones negativas")
print("✓ Baseline evaluado sobre los mismos 15 días")

✓ Todas las filas de validación tienen predicción
✓ Sin predicciones negativas
✓ Baseline evaluado sobre los mismos 15 días


In [18]:
model_results = pd.DataFrame([
    {
        "model": "Seasonal baseline",
        "RMSLE": baseline_rmsle,
        "MAE": baseline_mae,
    }
])

model_results

,model,RMSLE,MAE
0,Seasonal baseline,0.536571,85.477984


In [19]:
MODEL_FEATURES = [
    # Identidad de la serie
    "store_nbr",
    "family",

    # Calendario
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "is_weekend",

    # Promoción
    "onpromotion",

    # Metadata de tienda
    "city",
    "state",
    "type",
    "cluster",

    # Feriados / eventos
    "is_holiday",
    "is_event",
    "is_transfer",
    "is_additional",
    "is_bridge",
    "is_work_day",
    "is_new_year_closure",
    "event_count",

    # Historia de demanda
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
]

CATEGORICAL_FEATURES = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type",
    "cluster",
]

TARGET = "sales"

print("Cantidad de features:", len(MODEL_FEATURES))
print("Categóricas:", len(CATEGORICAL_FEATURES))

Cantidad de features: 28
Categóricas: 6


In [20]:
train_features = features_full[
    features_full["date"] < validation_start
].copy()

X_train = train_features[MODEL_FEATURES].copy()
y_train = train_features[TARGET].copy()

for col in CATEGORICAL_FEATURES:
    X_train[col] = X_train[col].astype("category")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train.head()

X_train: (2671272, 28)
y_train: (2671272,)


,store_nbr,family,year,month,day,day_of_week,week_of_year,is_weekend,onpromotion,city,...,is_work_day,is_new_year_closure,event_count,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,25,LAWN AND GARDEN,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25,HOME CARE,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,25,HOME APPLIANCES,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25,HOME AND KITCHEN II,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25,HOME AND KITCHEN I,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
assert len(X_train) == len(y_train)

assert "sales" not in X_train.columns
assert "date" not in X_train.columns
assert "tipo_cero" not in X_train.columns
assert "familia_surtida" not in X_train.columns
assert "tienda_operativa" not in X_train.columns

assert train_features["date"].max() == pd.Timestamp("2017-07-31")

print("✓ Target separado")
print("✓ Variables diagnósticas excluidas")
print("✓ Fecha cruda excluida")
print("✓ Train termina el 31/07/2017")
print("✓ Sin información real de agosto en entrenamiento")

✓ Target separado
✓ Variables diagnósticas excluidas
✓ Fecha cruda excluida
✓ Train termina el 31/07/2017
✓ Sin información real de agosto en entrenamiento


In [22]:
missing_features = (
    X_train
    .isna()
    .mean()
    .sort_values(ascending=False)
)

missing_features[
    missing_features > 0
]

lag_28             0.020534
lag_14             0.011472
lag_7              0.006941
lag_1              0.003058
rolling_mean_28    0.000647
rolling_mean_14    0.000647
rolling_mean_7     0.000647
dtype: float64

In [23]:
%pip install lightgbm==4.7.0

Note: you may need to restart the kernel to use updated packages.


In [24]:
import lightgbm as lgb

print("LightGBM:", lgb.__version__)

LightGBM: 4.7.0


In [25]:
y_train_log = np.log1p(y_train)

lgbm_model = lgb.LGBMRegressor(
    objective="regression",

    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,

    min_child_samples=100,

    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

lgbm_model.fit(
    X_train,
    y_train_log,
    categorical_feature=CATEGORICAL_FEATURES,
)

print("✓ LightGBM entrenado")

✓ LightGBM entrenado


In [26]:
print("Árboles:", lgbm_model.booster_.num_trees())
print("Features:", lgbm_model.booster_.num_feature())

assert lgbm_model.booster_.num_feature() == len(MODEL_FEATURES)

print("✓ Modelo listo para forecasting")

Árboles: 500
Features: 28
✓ Modelo listo para forecasting


In [27]:
from src.models.recursive_forecast import recursive_forecast

recursive_predictions = recursive_forecast(
    model=lgbm_model,
    features_base=features_base,
    train=train,
    validation_start=validation_start,
    validation_end=validation_end,
    model_features=MODEL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

recursive_predictions.head()

,date,store_nbr,family,prediction
0,2017-08-01,41,PRODUCE,2112.608363
1,2017-08-01,41,SCHOOL AND OFFICE SUPPLIES,0.479058
2,2017-08-01,41,SEAFOOD,6.482820
3,2017-08-01,42,AUTOMOTIVE,3.661890
4,2017-08-01,42,BREAD/BAKERY,498.183071


In [28]:
print(
    "Predicciones:",
    len(recursive_predictions),
)

print(
    "Días:",
    recursive_predictions["date"].nunique(),
)

print(
    "Desde:",
    recursive_predictions["date"].min().date(),
)

print(
    "Hasta:",
    recursive_predictions["date"].max().date(),
)

Predicciones: 25935
Días: 15
Desde: 2017-08-01
Hasta: 2017-08-15


In [29]:
validation_truth = validation[
    ["date", "store_nbr", "family", "sales"]
].copy()

lgbm_eval = validation_truth.merge(
    recursive_predictions,
    on=["date", "store_nbr", "family"],
    how="left",
    validate="one_to_one",
)

assert len(lgbm_eval) == len(validation)
assert lgbm_eval["prediction"].notna().all()
assert np.isfinite(lgbm_eval["prediction"]).all()
assert (lgbm_eval["prediction"] >= 0).all()

print("✓ Todas las predicciones emparejadas")
print("✓ Sin NaN ni infinitos")
print("✓ Sin predicciones negativas")

✓ Todas las predicciones emparejadas
✓ Sin NaN ni infinitos
✓ Sin predicciones negativas


In [30]:
lgbm_rmsle = rmsle(
    lgbm_eval["sales"],
    lgbm_eval["prediction"],
)

lgbm_mae = np.mean(
    np.abs(
        lgbm_eval["sales"]
        - lgbm_eval["prediction"]
    )
)

print(f"RMSLE LightGBM: {lgbm_rmsle:.4f}")
print(f"MAE LightGBM:   {lgbm_mae:.4f}")

RMSLE LightGBM: 0.4059
MAE LightGBM:   70.2984


In [31]:
model_results = pd.DataFrame([
    {
        "model": "Seasonal baseline",
        "RMSLE": baseline_rmsle,
        "MAE": baseline_mae,
    },
    {
        "model": "LightGBM recursive",
        "RMSLE": lgbm_rmsle,
        "MAE": lgbm_mae,
    },
])

model_results["RMSLE_vs_baseline_pct"] = (
    (baseline_rmsle - model_results["RMSLE"])
    / baseline_rmsle
    * 100
)

model_results

,model,RMSLE,MAE,RMSLE_vs_baseline_pct
0,Seasonal baseline,0.536571,85.477984,0.000000
1,LightGBM recursive,0.405895,70.298360,24.353902


In [32]:
lgbm_eval = lgbm_eval.copy()

lgbm_eval["horizon_day"] = (
    lgbm_eval["date"] - validation_start
).dt.days + 1

daily_diagnostics = (
    lgbm_eval
    .groupby(["horizon_day", "date"], as_index=False)
    .apply(
        lambda g: pd.Series({
            "RMSLE": rmsle(
                g["sales"],
                g["prediction"],
            ),
            "MAE": np.mean(
                np.abs(
                    g["sales"]
                    - g["prediction"]
                )
            ),
        }),
        include_groups=False,
    )
)

daily_diagnostics

,horizon_day,date,RMSLE,MAE
0,1,2017-08-01,0.370895,65.841681
1,2,2017-08-02,0.385975,62.398946
2,3,2017-08-03,0.396325,65.805973
3,4,2017-08-04,0.418790,69.598600
4,5,2017-08-05,0.379034,72.824848
5,6,2017-08-06,0.398073,79.429911
6,7,2017-08-07,0.403971,54.258635
7,8,2017-08-08,0.420668,59.103388
8,9,2017-08-09,0.393556,53.913524
9,10,2017-08-10,0.410466,57.249101


In [33]:
import plotly.express as px

fig = px.line(
    daily_diagnostics,
    x="horizon_day",
    y="RMSLE",
    markers=True,
    title="RMSLE por día del horizonte de pronóstico",
)

fig.update_layout(
    xaxis_title="Día del horizonte",
    yaxis_title="RMSLE",
)

fig.show()

In [34]:
fig = px.line(
    daily_diagnostics,
    x="horizon_day",
    y="MAE",
    markers=True,
    title="MAE por día del horizonte de pronóstico",
)

fig.update_layout(
    xaxis_title="Día del horizonte",
    yaxis_title="MAE",
)

fig.show()

In [35]:
store_diagnostics = (
    lgbm_eval
    .groupby("store_nbr", as_index=False)
    .apply(
        lambda g: pd.Series({
            "RMSLE": rmsle(
                g["sales"],
                g["prediction"],
            ),
            "MAE": np.mean(
                np.abs(
                    g["sales"]
                    - g["prediction"]
                )
            ),
            "n_obs": len(g),
        }),
        include_groups=False,
    )
    .sort_values("RMSLE", ascending=False)
)

store_diagnostics.head(10)

,store_nbr,RMSLE,MAE,n_obs
49,50,0.530779,115.794235,480.0
18,19,0.528301,34.545873,480.0
25,26,0.486488,36.678719,495.0
47,48,0.482344,132.818017,480.0
21,22,0.481930,45.925184,480.0
24,25,0.475832,65.825947,480.0
13,14,0.472729,62.947566,465.0
35,36,0.471045,72.058166,480.0
42,43,0.457811,62.900497,465.0
53,54,0.452004,82.744201,450.0


In [36]:
family_diagnostics = (
    lgbm_eval
    .groupby("family", as_index=False)
    .apply(
        lambda g: pd.Series({
            "RMSLE": rmsle(
                g["sales"],
                g["prediction"],
            ),
            "MAE": np.mean(
                np.abs(
                    g["sales"]
                    - g["prediction"]
                )
            ),
            "n_obs": len(g),
        }),
        include_groups=False,
    )
    .sort_values("RMSLE", ascending=False)
)

family_diagnostics.head(15)

,family,RMSLE,MAE,n_obs
31,SCHOOL AND OFFICE SUPPLIES,0.666652,38.678697,810.0
21,LINGERIE,0.623653,3.968030,810.0
13,GROCERY II,0.621487,11.275666,810.0
19,LADIESWEAR,0.569231,4.676351,660.0
23,MAGAZINES,0.547509,3.055557,810.0
6,CELEBRATION,0.542109,4.923143,810.0
14,HARDWARE,0.531223,1.070760,810.0
0,AUTOMOTIVE,0.505857,2.873858,810.0
15,HOME AND KITCHEN I,0.490386,11.108009,810.0
32,SEAFOOD,0.486205,4.317709,810.0


In [37]:
fig = px.bar(
    store_diagnostics.head(15),
    x="store_nbr",
    y="RMSLE",
    title="15 tiendas con mayor RMSLE",
)

fig.show()

In [38]:
fig = px.bar(
    family_diagnostics.head(15),
    x="family",
    y="RMSLE",
    title="15 familias con mayor RMSLE",
)

fig.show()

In [39]:
from src.features.demand_classification import build_demand_profile

train_demand_profile = build_demand_profile(
    train
)

train_demand_profile[
    [
        "store_nbr",
        "family",
        "adi",
        "cv2",
        "tipo_demanda",
    ]
].head()

,store_nbr,family,adi,cv2,tipo_demanda
0,25,LAWN AND GARDEN,1.128465,1.792538,Errática
1,25,HOME CARE,1.727743,0.313338,Intermitente
2,25,HOME APPLIANCES,9.647399,0.179169,Intermitente
3,25,HOME AND KITCHEN II,1.555452,1.478721,Lumpy
4,25,HOME AND KITCHEN I,1.539668,5.001265,Lumpy


In [40]:
lgbm_eval_demand = lgbm_eval.merge(
    train_demand_profile[
        [
            "store_nbr",
            "family",
            "tipo_demanda",
        ]
    ],
    on=["store_nbr", "family"],
    how="left",
    validate="many_to_one",
)

assert lgbm_eval_demand["tipo_demanda"].notna().all()

print("✓ Todas las predicciones tienen tipo de demanda")

✓ Todas las predicciones tienen tipo de demanda


In [41]:
demand_diagnostics = (
    lgbm_eval_demand
    .groupby("tipo_demanda", as_index=False)
    .apply(
        lambda g: pd.Series({
            "RMSLE": rmsle(
                g["sales"],
                g["prediction"],
            ),
            "MAE": np.mean(
                np.abs(
                    g["sales"]
                    - g["prediction"]
                )
            ),
            "n_obs": len(g),
            "series": g[
                ["store_nbr", "family"]
            ].drop_duplicates().shape[0],
        }),
        include_groups=False,
    )
    .sort_values("RMSLE")
)

demand_diagnostics

,tipo_demanda,RMSLE,MAE,n_obs,series
3,Regular,0.284044,123.183028,11415.0,761.0
1,Intermitente,0.402845,16.801800,6135.0,409.0
0,Errática,0.495667,61.171192,4320.0,288.0
2,Lumpy,0.564394,12.230041,4065.0,271.0


In [42]:
fig = px.bar(
    demand_diagnostics,
    x="tipo_demanda",
    y="RMSLE",
    text="RMSLE",
    title="RMSLE de LightGBM según tipo de demanda",
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
)

fig.show()

In [43]:
feature_importance = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "gain": lgbm_model.booster_.feature_importance(
        importance_type="gain"
    ),
    "splits": lgbm_model.booster_.feature_importance(
        importance_type="split"
    ),
})

feature_importance["gain_pct"] = (
    feature_importance["gain"]
    / feature_importance["gain"].sum()
    * 100
)

feature_importance = (
    feature_importance
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

feature_importance.head(15)

,feature,gain,splits,gain_pct
0,rolling_mean_7,1.084941e+08,2250,57.615671
1,lag_7,6.391981e+07,1800,33.944544
2,lag_1,8.927050e+06,2200,4.740700
3,rolling_mean_14,2.328336e+06,590,1.236460
4,lag_14,1.079153e+06,1136,0.573083
5,family,1.009838e+06,5293,0.536273
6,is_new_year_closure,5.418158e+05,267,0.287731
7,lag_28,4.378186e+05,1106,0.232503
8,day_of_week,3.698523e+05,1891,0.196410
9,day,2.145109e+05,2088,0.113916


In [44]:
importance_plot = (
    feature_importance
    .head(15)
    .sort_values("gain_pct")
)

fig = px.bar(
    importance_plot,
    x="gain_pct",
    y="feature",
    orientation="h",
    title="15 features más importantes — LightGBM",
)

fig.update_layout(
    xaxis_title="% de ganancia total",
    yaxis_title="Feature",
)

fig.show()

In [45]:
first_5_rmsle = daily_diagnostics.loc[
    daily_diagnostics["horizon_day"].le(5),
    "RMSLE",
].mean()

last_5_rmsle = daily_diagnostics.loc[
    daily_diagnostics["horizon_day"].ge(11),
    "RMSLE",
].mean()

rmsle_improvement = (
    (baseline_rmsle - lgbm_rmsle)
    / baseline_rmsle
    * 100
)

mae_improvement = (
    (baseline_mae - lgbm_mae)
    / baseline_mae
    * 100
)

print("RESULTADO GENERAL")
print(f"Baseline RMSLE: {baseline_rmsle:.4f}")
print(f"LightGBM RMSLE: {lgbm_rmsle:.4f}")
print(f"Mejora RMSLE:   {rmsle_improvement:.2f}%")

print()

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"LightGBM MAE: {lgbm_mae:.4f}")
print(f"Mejora MAE:   {mae_improvement:.2f}%")

print()

print("HORIZONTE")
print(f"RMSLE primeros 5 días: {first_5_rmsle:.4f}")
print(f"RMSLE últimos 5 días:  {last_5_rmsle:.4f}")

print()

print("TIPO DE DEMANDA")
display(
    demand_diagnostics[
        ["tipo_demanda", "RMSLE", "MAE", "series"]
    ]
)

print("FEATURES PRINCIPALES")
display(
    feature_importance[
        ["feature", "gain_pct"]
    ].head(10)
)

RESULTADO GENERAL
Baseline RMSLE: 0.5366
LightGBM RMSLE: 0.4059
Mejora RMSLE:   24.35%

Baseline MAE: 85.4780
LightGBM MAE: 70.2984
Mejora MAE:   17.76%

HORIZONTE
RMSLE primeros 5 días: 0.3902
RMSLE últimos 5 días:  0.4210

TIPO DE DEMANDA


,tipo_demanda,RMSLE,MAE,series
3,Regular,0.284044,123.183028,761.0
1,Intermitente,0.402845,16.801800,409.0
0,Errática,0.495667,61.171192,288.0
2,Lumpy,0.564394,12.230041,271.0


FEATURES PRINCIPALES


,feature,gain_pct
0,rolling_mean_7,57.615671
1,lag_7,33.944544
2,lag_1,4.740700
3,rolling_mean_14,1.236460
4,lag_14,0.573083
5,family,0.536273
6,is_new_year_closure,0.287731
7,lag_28,0.232503
8,day_of_week,0.196410
9,day,0.113916


In [46]:
assert lgbm_rmsle < baseline_rmsle
assert lgbm_mae < baseline_mae

print("✓ LightGBM supera al baseline en RMSLE")
print("✓ LightGBM supera al baseline en MAE")
print("✓ Forecast evaluado recursivamente durante 15 días")
print("✓ Diagnóstico por horizonte, tienda, familia y ADI/CV² completado")

✓ LightGBM supera al baseline en RMSLE
✓ LightGBM supera al baseline en MAE
✓ Forecast evaluado recursivamente durante 15 días
✓ Diagnóstico por horizonte, tienda, familia y ADI/CV² completado
